# שיעור 04 - תבנית עיצוב לשימוש בכלים

בשיעור זה תלמדו את תבנית העיצוב **שימוש בכלים** לסוכני AI באמצעות Microsoft Agent Framework (Python). אנו מכסים את הנושאים הבאים:

- הגדרת כלים כפונקציות עם הדקורטור `@tool` ופרמטרים ממופים
- מתן סכימות לכלים כך שהמודל ידע מה כל כלי עושה
- שליטה בביצוע הכלי עם `approval_mode`
- החזרת **פלט מובנה** באמצעות מודלים של Pydantic ו-`response_format`

התסריט הוא של **סוכן הזמנות טיולים** שיכול לבדוק יעדים, לבדוק זמינות, ולאחזר מידע על טיסות.


## הגדרה


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## הגדרת כלים עם הדקורטור @tool

הדקורטור `@tool` הופך פונקציית פייתון פשוטה לכלי שסוכן יכול לקרוא לו.  
נקודות מפתח:

- ה**docstring** הופך לתיאור הכלי שהמודל רואה.  
- **הערות סוג** (כולל `Annotated` עם תיאורים) מגדירות את סכימת הכלי.  
- `approval_mode` שולט אם המשתמש חייב לאשר כל קריאה לפני שהיא מתבצעת.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## יצירת סוכן עם כלים מרובים

העבר את כל שלושת הכלים ללקוח כך שהמודל יוכל לקרוא לכל אחד מהם לפי הצורך כדי לענות על שאלת המשתמש.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## פלט מובנה עם כלים

על ידי הגדרת `response_format` למודל Pydantic, הסוכן מוכרח להחזיר אובייקט JSON טיפוסי היטב במקום טקסט חופשי. זה שימושי כאשר קוד להמשך צריך לצרוך את התוצאה באופן תכנותי.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## דפוסי אישור כלים

הפרמטר `approval_mode` על `@tool` שולט האם קריאות לכלי דורשות אישור אנושי לפני ביצוע:

| מצב | התנהגות |
|---|---|
| `"never_require"` | הכלי רץ באופן אוטומטי — אין צורך באישור משתמש. |
| `"always_require"` | כל קריאה חייבת להיות מאושרת על ידי המשתמש לפני ביצועה. |

השתמש ב-`"always_require"` לכלים שיש להם תוצאות צדדיות (למשל הזמנת טיסה, חיוב כרטיס אשראי) כדי שאדם יישאר בלולאה.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## סיכום

בשיעור זה למדת כיצד:

1. **להגדיר כלים** באמצעות הדקורטור `@tool` עם פרמטרים ממופים ו- docstrings המשמשים כסכימת הכלי.
2. **לרכוב מספר כלים** כך שהסוכן יכול לקרוא להם ברצף כדי לענות על שאילתות מורכבות.
3. **להחזיר פלט מובנה** על ידי העברת מודל Pydantic כ-`response_format`.
4. **לשלוט באישור כלי** עם `approval_mode` כדי לשמור על מעורבות אדם בפעולות רגישות.

תבניות אלה מהוות את הבסיס לבניית סוכנים אמינים, מוכנים לייצור, שיכולים לתקשר בבטחה עם מערכות חיצוניות.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:  
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). בעוד שאנו שואפים לדיוק, יש להביא בחשבון כי תרגומים אוטומטיים עשויים להכיל שגיאות או אי-דיוקים. המסמך המקורי בשפתו המקורית נחשב למקור הסמכותי. עבור מידע קריטי, מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אנושי. איננו נושאים באחריות על כל אי-הבנות או פרשנויות שגויות הנובעות מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
